# Qwen3-4B Sinhala QA Fine-Tuning on the CPT checkpoint (corrected)

Fine-tunes **`isji/qwen3-4b-cpt`** (Sinhala CPT + `isji/Extended-Sinhala-Qwen3`,
176,856 tokens) for grounded Sinhala QA. This replaces the earlier CPT+QA run that
produced `isji/qwen3-4b-sinhala-qa-cpt-merged`, which scored far below both stock Qwen
and the no-CPT QA model.

## What went wrong last time, and what this notebook changes

Measured on the 153-row test split:

| System | Exact match | Token F1 | Answerable EM | Unanswerable | Over-refusal |
|---|---|---|---|---|---|
| `Qwen3-4B-Instruct-2507` (stock) | 20.26% | 0.5048 | 12.59% | 77.8% | 0% |
| `qwen3-4b-sinhala-qa-poc-merged` (QA-FT, no CPT) | 20.13% | 0.5015 | 12.50% | 77.8% | 0% |
| `qwen3-4b-sinhala-qa-cpt-merged` **v1 (broken)** | 12.42% | 0.2414 | **0.74%** | 100% | **50.4%** |
| `qwen3-4b-sinhala-qa-cpt-v2-merged` **(this notebook)** | **28.76%** | **0.6161** | **19.26%** | **100%** | **6.7%** |

**The termination fixes worked.** v2 beats stock Qwen on every axis — +8.5 points of exact
match, +0.11 token F1, and it keeps a perfect unanswerable score (100%, 0% false answers)
where stock fabricates an answer on 22% of unanswerable questions. For scale, the Llama
route (SinLlama 3B) sits at EM 32.03% / F1 0.6504.

**The CPT model was not short on knowledge — it never learned to stop.** It emits the
correct answer and then keeps generating web-corpus text (marketing copy, news
articles) straight from the CPT data. Truncating those generations at the point they
stop being grounded raises token F1 from 0.275 to **0.559 on the rows it attempted —
above stock Qwen's 0.505**. The answer was always there, buried under filler.

Root cause: CPT packed raw documents separated by `<|endoftext|>` (151643), so it taught
*"Sinhala text continues; the only terminator is `<|endoftext|>`."* The model never saw
`<|im_end|>` (151645) — the chat/answer terminator — across 179.6M CPT tokens. The
previous QA fine-tune then tried to teach `<|im_end|>` with 1,375 examples over 3 epochs.
The CPT signal won.

The 50.4% over-refusal is the same bug seen through the grounding gate: the gate scores
`evidence_support` over the **whole** generation, so the ungrounded filler dragged support
below threshold and the gate rewrote correct answers as refusals.

### Fixes in this notebook

1. **`MAX_NEW_TOKENS` 320 → 64.** The 320 budget was derived for Qwen's *default*
   tokenizer (9.19 tokens/word). Under the extended tokenizer (1.48 tokens/word) the
   longest gold answer is 55 tokens, so 320 handed the model five times more rope than
   the task needs. This alone caps how far a non-terminating generation can run.
2. **Stronger termination training**: 8 epochs (was 3) and LoRA r=64/α=128 (was r=32/α=64),
   so the QA adapter has enough capacity and exposure to override the CPT continuation
   habit. Early stopping on eval loss guards against overfitting the 1,375 rows.
3. **An explicit EOS-supervision audit** before training starts: every completion is
   verified to tokenize with `<|im_end|>` as its final token, so the stop signal is
   definitely inside the completion-only loss span rather than silently absent.
4. **An EOS-emission rate metric at evaluation.** If the model still fails to terminate,
   this reports it as a number instead of leaving it to be inferred from bad scores.
5. **Grounded-prefix cleanup as a safety net**, applied *before* the grounding gate
   scores the answer. It keeps the generation up to the first run of consecutive
   ungrounded words. Validated to leave Sinhala abbreviations intact
   (`ක්‍රි.ව. 1595 වර්ෂයේ සිට ය.`, `සී. ඩබ්ලිව්. ඩබ්ලිව්. කන්නන්ගර මහතා ය.` survive
   unchanged), which naive sentence-splitting on `.` would destroy.
6. **The gate scores the cleaned answer**, which is what removes the 50% over-refusal.

### Accuracy pass added after the v2 run

Analysis of the v2 output showed where the remaining points sit, so this version adds:

- **A. Beam search (`NUM_BEAMS = 4`), was greedy.** 18 answerable rows differed from the
  reference in the **first word only**, with the following words matching exactly
  (`දොන්ජොලේ මාර්ගවල` for `ඇළ මාර්ගවල`, `අපට` for `මීට`, `ජෙනරල් වොට්` for `ජේම්ස් වොට්`).
  Greedy decoding commits to a bad opening token and cannot back out; beam search can.
- **B. Grounding threshold 0.50 → 0.40.** v2 still refused 9/135 answerable questions while
  holding 100% unanswerable accuracy at a 0% false-answer rate — the abstention side had
  margin and the gate was the binding constraint on recall.
- **C. 6 epochs** (the v2 run used 3). Not higher: on this same 1,370-example dataset the
  Llama 1B 5-epoch run peaked at epoch 2 and then overfit. Early stopping plus
  `load_best_model_at_end` mean the extra budget costs time, not quality — and evaluation
  now runs ~2x per epoch so early stopping can actually catch the turn.
- **D. Style-insensitive exact match reported alongside strict EM.** Sinhala answers end in
  a contentless grammatical particle: the model writes `රෝලට් පනත යි.` where the reference
  says `රෝලට් පනත ය.` — token F1 = 1.00, strict EM = miss. Strict EM stays the headline
  number (it is what remains comparable with the Llama v6 results); this is a diagnostic
  that separates content errors from inflection differences.

### Carried over from the previous notebook (still required)

`isji/qwen3-4b-cpt` is a LoRA adapter, not a merged model. Qwen3-4B ties
`embed_tokens`/`lm_head` into one matrix, and PEFT's `modules_to_save` silently unties
them on **every fresh load** — the trainable copy feeds the input side while `lm_head`
keeps the frozen, mean-initialised original. Left unfixed, the 25,187 new Sinhala tokens
are readable but ungeneratable. The model cell re-ties the head immediately after
loading, asserts it before merging, and verifies the merged result by value.

In [ ]:
%uv pip install -q "transformers>=4.51,<5" "trl>=0.26,<0.30" "peft>=0.19" datasets accelerate huggingface_hub hf_transfer tqdm

In [ ]:
import hashlib
import json
import math
import os
import random
import re
import unicodedata
from collections import Counter
from pathlib import Path

import numpy as np
import torch
from tqdm.auto import tqdm
from transformers import set_seed

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

SEED = 42

# ---- Model ----
BASE_MODEL_ID = "Qwen/Qwen3-4B"
CPT_ADAPTER_ID = "isji/qwen3-4b-cpt"      # Sinhala CPT adapter + extended tokenizer

# ---- Data ----
TRAIN_CANDIDATES = [
    Path("/tmp/train.jsonl"),
    Path("new_split_v2/train.jsonl"),
    Path("../new_split_v2/train.jsonl"),
]
TRAIN_PATH = next((p for p in TRAIN_CANDIDATES if p.is_file()), TRAIN_CANDIDATES[0])
TEST_CANDIDATES = [
    Path("/tmp/test_updated.jsonl"),
    Path("/tmp/test.jsonl"),
    Path("new_split_v2/test.jsonl"),
    Path("../new_split_v2/test.jsonl"),
]
TEST_PATH = next((p for p in TEST_CANDIDATES if p.is_file()), TEST_CANDIDATES[0])

# ---- Output ----
OUTPUT_DIR = Path("/tmp/output_qwen3_4b_qa_cpt_v2")
QA_ADAPTER_DIR = Path("/tmp/qwen3_4b_qa_cpt_v2_adapter")
MERGED_MODEL_DIR = Path("/tmp/qwen3_4b_qa_cpt_v2_merged")
RESULTS_JSONL = Path("/tmp/qwen3_4b_qa_cpt_v2_results.jsonl")
RESULTS_TXT = Path("/tmp/qwen3_4b_qa_cpt_v2_results.txt")
MERGED_REPO_ID = os.environ.get("QA_MERGED_REPO_ID", "isji/qwen3-4b-sinhala-qa-cpt-v2-merged")
ADAPTER_REPO_ID = os.environ.get("QA_ADAPTER_REPO_ID", "isji/qwen3-4b-sinhala-qa-cpt-v2-adapter")
HF_REPO_PRIVATE = True

NO_ANSWER = "මෙම ප්‍රශ්නයට පිළිතුරු දීමට ප්‍රමාණවත් තොරතුරු නොමැත."
MAX_LENGTH = 2048

# FIX 1: budget matched to the EXTENDED tokenizer, not the default one.
# Longest gold answer is 55 tokens under isji/Extended-Sinhala-Qwen3; 320 (the old value,
# derived for the 9.19 tokens/word default tokenizer) gave a non-terminating model five
# times more room to ramble than the task can ever need. Re-audited against the data below.
MAX_NEW_TOKENS = 64

# FIX 2: enough capacity and exposure to override the CPT continuation habit.
#
# Epoch count — read this before changing it. The v2 run that reached EM 28.76% / F1 0.6161
# and fixed termination completely used **3 epochs**, so 3 is a proven-good floor rather
# than a guess. Evidence from the Llama route on this same 1,370-example dataset says not to
# go far past it: the 1B 5-epoch run hit its best validation loss at step 88 of 220 (epoch 2)
# and then rose — 0.7046 -> 0.7231 -> 0.7092 -> 0.7125 — while training loss kept falling
# 0.495 -> 0.239, i.e. memorising. The 3B 3-epoch run peaked at step 117 of 132 and ticked
# up at 130.
#
# 6 gives the answer-style alignment room to improve past 3 without inviting that
# overfitting, and the run is protected on both sides: EarlyStoppingCallback (patience 4)
# halts once eval loss stops improving, and load_best_model_at_end restores the best
# checkpoint regardless of where training ended. So if ~3 epochs really is optimal, the
# extra budget costs wall-clock time, not model quality.
#
# Watch the eval-loss column in the training log: if it bottoms out in the first 3 epochs
# and only rises after, drop this back to 3 or 4 for the next run.
NUM_EPOCHS = 6
LORA_R = 64
LORA_ALPHA = 128
LEARNING_RATE = 2e-4

# ---- ACCURACY FIX A: beam search instead of greedy ----
# Measured on the first corrected run: 18 answerable rows differ from the reference in the
# FIRST WORD ONLY, with the following words matching exactly — e.g. "දොන්ජොලේ මාර්ගවල" for
# "ඇළ මාර්ගවල", "අපට" for "මීට", "ජෙනරල් වොට්" for "ජේම්ස් වොට්". Greedy decoding commits to
# a bad opening token and cannot recover. Beam search keeps alternatives alive long enough
# for the better-scoring continuation to win. Set NUM_BEAMS = 1 to revert to greedy.
NUM_BEAMS = 4
LENGTH_PENALTY = 1.0

VALIDATION_FRACTION = 0.02
MIN_UNANSWERABLE_TRAIN_FRACTION = 0.25

# ---- ACCURACY FIX B: gate relaxed 0.50 -> 0.40 ----
# The first corrected run still refused 9/135 answerable questions while sitting at 100%
# unanswerable accuracy with a 0% false-answer rate — i.e. the abstention side had margin to
# spare and the gate was the binding constraint on recall. Watch "Gated to refusal" and the
# false-answer rate in the summary: if false answers appear, raise this back toward 0.50.
GROUNDING_THRESHOLD = 0.40
USE_GROUNDING = True
USE_FULL_CONTEXT = True

# FIX 5: grounded-prefix cleanup. Cut the generation at the first run of this many
# consecutive words that are not supported by the context. Set to 0 to disable.
UNGROUNDED_RUN_TO_CUT = 3

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

print("Base model     :", BASE_MODEL_ID)
print("CPT adapter    :", CPT_ADAPTER_ID)
print("Training data  :", TRAIN_PATH, "| exists:", TRAIN_PATH.is_file())
print("External test  :", TEST_PATH, "| exists:", TEST_PATH.is_file())
print(f"Epochs         : {NUM_EPOCHS} | LoRA r={LORA_R} alpha={LORA_ALPHA} | lr={LEARNING_RATE}")
print(f"MAX_NEW_TOKENS : {MAX_NEW_TOKENS}")

In [ ]:
# Required: the CPT adapter repo is private. Reads HF_TOKEN from the environment — never
# paste a token into this notebook; a committed token is a leaked credential.
from huggingface_hub import login

_token = os.environ.get("HF_TOKEN")
if _token:
    login(token=_token, add_to_git_credential=False)
    print("Logged in to Hugging Face from HF_TOKEN.")
else:
    print("WARNING: no HF_TOKEN in the environment — loading a private adapter will fail.")

In [ ]:
from transformers import AutoTokenizer

# The CPT adapter ships the extended tokenizer (176,856 tokens) with eos aligned to
# <|im_end|>. Loading it from anywhere else would desynchronise the token IDs.
tokenizer = AutoTokenizer.from_pretrained(CPT_ADAPTER_ID, use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
tokenizer.truncation_side = "right"

TEMPLATE_SUPPORTS_THINKING = "enable_thinking" in (tokenizer.chat_template or "")
CHAT_KWARGS = {"enable_thinking": False} if TEMPLATE_SUPPORTS_THINKING else {}

if tokenizer.eos_token_id != 151645:
    raise RuntimeError(
        f"Expected eos <|im_end|> (151645) for chat-format QA training, got "
        f"{tokenizer.eos_token!r} ({tokenizer.eos_token_id}). Training would teach the "
        "wrong stop token."
    )

print("Tokenizer      :", CPT_ADAPTER_ID)
print("Vocabulary     :", f"{len(tokenizer):,}")
print("eos / pad      :", f"{tokenizer.eos_token!r} ({tokenizer.eos_token_id}) / "
      f"{tokenizer.pad_token!r} ({tokenizer.pad_token_id})")
print("Thinking switch:", "present -> disabled" if TEMPLATE_SUPPORTS_THINKING else "not in template")

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required for this notebook.")
train_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

print(f"Loading base model {BASE_MODEL_ID}...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    dtype=train_dtype,
    device_map={"": torch.cuda.current_device()},
    attn_implementation="sdpa",
    low_cpu_mem_usage=True,
)

original_vocab_size = model.get_input_embeddings().num_embeddings
if original_vocab_size != len(tokenizer):
    model.resize_token_embeddings(len(tokenizer), mean_resizing=True)
    print(f"Resized embeddings: {original_vocab_size:,} -> {len(tokenizer):,}")

input_weight = model.get_input_embeddings().weight
output_weight = model.get_output_embeddings().weight
embeddings_are_tied = input_weight.data_ptr() == output_weight.data_ptr()
print("Embeddings tied after resize:", embeddings_are_tied)

print(f"Loading CPT adapter {CPT_ADAPTER_ID}...")
model = PeftModel.from_pretrained(model, CPT_ADAPTER_ID)

# ---- Tied-embedding fix: PEFT unties the head on every fresh load ----
if embeddings_are_tied:
    base_model = model.base_model.model
    embed_wrapper = base_model.model.embed_tokens
    if hasattr(embed_wrapper, "modules_to_save"):
        trained_embedding = embed_wrapper.modules_to_save["default"].weight
        if base_model.lm_head.weight.data_ptr() != trained_embedding.data_ptr():
            base_model.lm_head.weight = embed_wrapper.modules_to_save["default"].weight
            print("Re-tied lm_head to the loaded, CPT-trained embedding copy.")
        if base_model.lm_head.weight.data_ptr() != trained_embedding.data_ptr():
            raise RuntimeError(
                "lm_head is not tied to the trained embedding copy — the 25,187 new Sinhala "
                "tokens would be unusable for generation. Do not proceed."
            )
        print("Tie verified before merge.")
        pre_merge_embedding = trained_embedding.detach().clone()
    else:
        pre_merge_embedding = None
else:
    pre_merge_embedding = None

print("Merging CPT adapter...")
model = model.merge_and_unload()

merged_input = model.get_input_embeddings().weight
merged_output = model.get_output_embeddings().weight
if merged_input.shape[0] != len(tokenizer):
    raise RuntimeError(
        f"Merged vocabulary ({merged_input.shape[0]:,}) != tokenizer ({len(tokenizer):,})."
    )
if not torch.equal(merged_input, merged_output):
    raise RuntimeError(
        "Merged input/output embeddings differ — lm_head did not receive the CPT-trained "
        "values. Do not train on this model."
    )
if pre_merge_embedding is not None and not torch.equal(
    merged_input.detach().cpu(), pre_merge_embedding.cpu()
):
    raise RuntimeError("Merged embeddings differ from the loaded CPT values.")
print(f"Merge verified: input == output, shape ({merged_input.shape[0]:,}, {merged_input.shape[1]:,}).")

model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()
print("CPT model ready for QA fine-tuning | dtype:", train_dtype)

In [ ]:
SYSTEM_PROMPT = f"""You are a helpful Sinhala history question-answering assistant.

Your task is to answer the question using ONLY the information explicitly provided in the context.

Instructions:

- Read the entire context carefully before answering.
- Use only the information explicitly stated in the context.
- Do not use external knowledge, assumptions, or prior knowledge.
- Identify the exact information requested by the question.
- If the answer is found in multiple parts of the context, combine the relevant information into a single complete answer.
- Include only information that directly answers the question.
- Do not include additional facts, names, dates, or events unless they are required to answer the question.
- Match the person or entity named in the question exactly.
- Use evidence that contains both the requested entity and the requested attribute.
- Do not take a date or fact from a neighboring sentence about a different entity or event.
- Do not infer or guess information that is not explicitly stated, except for simple arithmetic explicitly requested by the question when all required values are stated in the context.
- For a duration question with explicit starting and ending years, subtract the starting year from the ending year and return the duration.
- If the answer cannot be found in the context, respond exactly with:
  "{NO_ANSWER}"
- Return only the final answer in natural Sinhala.
- Do not explain your reasoning.
- Do not mention passage numbers, page numbers, chapter names, grades, or any other source references.
- Answer in a single line, then stop. Do not continue with any further text."""

SINHALA_WORD_RE = re.compile(r"[\w඀-෿]+", re.UNICODE)
STOPWORDS = {
    "හා", "සහ", "හෝ", "දී", "ද", "ය", "යි", "වේ", "විය", "වූ", "ලෙස",
    "විසින්", "සඳහා", "සිට", "දක්වා", "එම", "මෙම", "ඒ", "ඔහු", "ඇය",
    "කුමක්ද", "කවුද", "කවදාද", "කෙසේද", "කොපමණද", "මොනවාද",
}


def clean_text(value):
    text = unicodedata.normalize("NFC", str(value or ""))
    return text.replace("\r\n", "\n").replace("\r", "\n").strip()


def lexical_tokens(value):
    tokens = [token.casefold() for token in SINHALA_WORD_RE.findall(clean_text(value))]
    return [token for token in tokens if len(token) >= 2 and token not in STOPWORDS]


def token_supported(token, normalized_context):
    if token in normalized_context:
        return True
    return len(token) >= 4 and token[:-1] in normalized_context


def evidence_support(answer, context):
    answer_tokens = lexical_tokens(answer)
    if not answer_tokens:
        return 0.0
    normalized_context = " ".join(lexical_tokens(context))
    supported = sum(token_supported(t, normalized_context) for t in answer_tokens)
    return supported / len(answer_tokens)


def build_messages(context, question):
    user_prompt = f"""Context:

{clean_text(context)}

Question:

{clean_text(question)}

Answer:"""
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]


def render_prompt(context, question):
    return tokenizer.apply_chat_template(
        build_messages(context, question),
        tokenize=False,
        add_generation_prompt=True,
        **CHAT_KWARGS,
    )


def prompt_token_count(text):
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])


def canonical_answer(item):
    if item.get("answerable") is False:
        return NO_ANSWER
    answer = clean_text(item.get("answer", ""))
    return answer if answer else NO_ANSWER


def load_jsonl(path):
    if not path.is_file():
        raise FileNotFoundError(f"Required JSONL file not found: {path}")
    records, fingerprints, dropped, duplicates = [], set(), 0, 0
    with path.open("r", encoding="utf-8-sig") as handle:
        for line_number, line in enumerate(handle, 1):
            if not line.strip():
                continue
            try:
                item = json.loads(line)
            except json.JSONDecodeError as error:
                raise ValueError(f"{path}:{line_number}: {error}") from error
            question = clean_text(item.get("question"))
            context = clean_text(item.get("context"))
            answerable = item.get("answerable")
            if type(answerable) is not bool:
                answerable = bool(clean_text(item.get("answer")))
            normalized = {
                "question": question,
                "context": context,
                "answer": clean_text(item.get("answer")),
                "answerable": answerable,
                "grade": item.get("grade"),
                "chapter": item.get("chapter"),
            }
            if not question or not context or (answerable and not normalized["answer"]):
                dropped += 1
                continue
            fp = (normalized["question"], normalized["context"], normalized["answer"], answerable)
            if fp in fingerprints:
                duplicates += 1
                continue
            fingerprints.add(fp)
            records.append(normalized)
    print(f"Loaded {len(records)} unique records from {path}")
    print(f"Dropped invalid/empty: {dropped}; exact duplicates removed: {duplicates}")
    return records


records = load_jsonl(TRAIN_PATH)
print(Counter(r["answerable"] for r in records))
print("\n--- rendered prompt tail ---")
print(render_prompt(records[0]["context"], records[0]["question"])[-400:])

In [ ]:
# ---- Build prompt/completion pairs, then audit EOS supervision ----

def make_training_example(item):
    answer = canonical_answer(item)
    completion = answer + tokenizer.eos_token      # <|im_end|> — the stop signal to learn
    prompt = render_prompt(item["context"], item["question"])
    return {"prompt": prompt, "completion": completion}


groups = {}
for item in records:
    key = hashlib.sha1(item["context"].encode("utf-8")).hexdigest()
    groups.setdefault(key, []).append(item)

group_keys = sorted(groups)
random.Random(SEED).shuffle(group_keys)
desired = max(200, round(len(group_keys) * VALIDATION_FRACTION))
cap = max(1, math.floor(len(group_keys) * 0.20))
validation_group_count = min(desired, cap, max(1, len(group_keys) - 1))
validation_keys = set(group_keys[:validation_group_count])

train_records, validation_records = [], []
for key, group in groups.items():
    (validation_records if key in validation_keys else train_records).extend(group)


def rebalance_unanswerable(items, minimum_fraction):
    positives = [i for i in items if i["answerable"]]
    negatives = [i for i in items if not i["answerable"]]
    if not negatives or len(negatives) / len(items) >= minimum_fraction:
        return list(items), 0
    required = math.ceil(minimum_fraction * len(positives) / (1.0 - minimum_fraction))
    extra = max(0, required - len(negatives))
    rng = random.Random(SEED)
    balanced = list(items) + [rng.choice(negatives) for _ in range(extra)]
    rng.shuffle(balanced)
    return balanced, extra


train_records, repeated_negatives = rebalance_unanswerable(
    train_records, MIN_UNANSWERABLE_TRAIN_FRACTION
)
print(f"Train records after balancing: {len(train_records):,} (+{repeated_negatives} repeated negatives)")
print(f"Validation records: {len(validation_records):,}")
print(f"Unanswerable fraction: {sum(1 for r in train_records if not r['answerable'])/len(train_records):.1%}")

train_examples = [make_training_example(i) for i in tqdm(train_records, desc="Train")]
validation_examples = [make_training_example(i) for i in tqdm(validation_records, desc="Validation")]

from datasets import Dataset

train_dataset = Dataset.from_list(train_examples)
validation_dataset = Dataset.from_list(validation_examples)

# ---- FIX 3: EOS supervision audit ----
# Every completion must tokenize with <|im_end|> as its final token. If this silently
# failed, the model would never receive a stop signal in the completion-only loss — which
# is exactly the failure the previous run exhibited.
bad_eos = 0
completion_lengths = []
for example in train_examples:
    ids = tokenizer(example["completion"], add_special_tokens=False)["input_ids"]
    completion_lengths.append(len(ids))
    if not ids or ids[-1] != tokenizer.eos_token_id:
        bad_eos += 1
completion_lengths.sort()

print("\n=== EOS supervision audit ===")
print(f"Completions ending in <|im_end|> ({tokenizer.eos_token_id}): "
      f"{len(train_examples) - bad_eos}/{len(train_examples)}")
if bad_eos:
    raise RuntimeError(
        f"{bad_eos} completions do not end with the eos token — the stop signal would not "
        "be trained. Fix make_training_example() before training."
    )
print(f"Completion tokens (incl. eos): median {completion_lengths[len(completion_lengths)//2]}, "
      f"p95 {completion_lengths[int(len(completion_lengths)*0.95)]}, max {completion_lengths[-1]}")
print(f"MAX_NEW_TOKENS = {MAX_NEW_TOKENS}")
if completion_lengths[-1] > MAX_NEW_TOKENS:
    print(f"WARNING: longest completion ({completion_lengths[-1]}) exceeds MAX_NEW_TOKENS "
          f"({MAX_NEW_TOKENS}) — raise it or answers will be cut off at generation time.")
else:
    print(f"OK: MAX_NEW_TOKENS covers every training answer with "
          f"{MAX_NEW_TOKENS - completion_lengths[-1]} tokens of headroom.")

lengths = [prompt_token_count(e["prompt"] + e["completion"]) for e in train_examples]
print(f"\nFull sequence tokens: max {max(lengths):,} / MAX_LENGTH {MAX_LENGTH:,}")
if max(lengths) > MAX_LENGTH - 4:
    raise RuntimeError("Some sequences exceed MAX_LENGTH; completion supervision could be truncated.")

In [ ]:
from peft import LoraConfig, TaskType
from transformers import EarlyStoppingCallback
from trl import SFTConfig, SFTTrainer

qa_lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

updates_per_epoch = math.ceil(len(train_dataset) / (4 * 8))
total_updates = max(1, updates_per_epoch * NUM_EPOCHS)

# Evaluate ~twice per epoch, independent of NUM_EPOCHS. Tying the interval to the total
# (the previous behaviour) made evaluation coarser as epochs grew — exactly backwards, since
# a longer run is when early stopping most needs the resolution to catch the turn.
eval_steps = max(10, updates_per_epoch // 2)
print(f"Estimated optimizer updates: {total_updates:,} ({updates_per_epoch}/epoch x {NUM_EPOCHS})")
print(f"Evaluation/save interval  : every {eval_steps} updates (~2x per epoch)")
print(f"Early stopping            : patience 4 evals (~{4 * eval_steps / updates_per_epoch:.1f} epochs "
      f"without improvement), best checkpoint restored at the end")

training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    max_length=MAX_LENGTH,
    completion_only_loss=True,      # loss on the answer + its eos, not the prompt
    packing=False,
    eval_packing=False,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=8,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    weight_decay=0.01,
    max_grad_norm=1.0,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    bf16=(train_dtype == torch.bfloat16),
    fp16=(train_dtype == torch.float16),
    optim="adamw_torch",
    eval_strategy="steps",
    eval_steps=eval_steps,
    save_strategy="steps",
    save_steps=eval_steps,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=10,
    logging_first_step=True,
    group_by_length=True,
    dataset_num_proc=min(8, os.cpu_count() or 1),
    report_to="none",
    seed=SEED,
    data_seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    processing_class=tokenizer,
    peft_config=qa_lora_config,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=4)],
)

trainer.model.print_trainable_parameters()
print("Starting QA fine-tuning on the Sinhala CPT checkpoint...")
train_result = trainer.train()
print(train_result)

In [ ]:
QA_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(QA_ADAPTER_DIR))
tokenizer.save_pretrained(QA_ADAPTER_DIR)

print("Merging the QA adapter...")
model = trainer.model.merge_and_unload()
model.config.use_cache = True
model.config.pad_token_id = tokenizer.pad_token_id
model.generation_config.pad_token_id = tokenizer.pad_token_id
model.generation_config.do_sample = False
model.generation_config.temperature = None
model.generation_config.top_p = None
model.generation_config.top_k = None
model.eval()

MERGED_MODEL_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(MERGED_MODEL_DIR, safe_serialization=True, max_shard_size="4GB")
tokenizer.save_pretrained(MERGED_MODEL_DIR)
print("QA adapter :", QA_ADAPTER_DIR)
print("Merged model:", MERGED_MODEL_DIR)

In [ ]:
# ---- Inference: generation, EOS tracking, grounded-prefix cleanup, grounding gate ----

def normalize_answer(value):
    text = clean_text(value).casefold()
    text = re.sub(r"\s+", " ", text)
    return text.strip(" \t\r\n[]{}()<>\"'`.,!?;:।෴")


def is_no_answer(value):
    normalized = normalize_answer(value)
    return normalized == normalize_answer(NO_ANSWER) or "ප්‍රමාණවත් තොරතුරු නොමැත" in normalized


# ---- ACCURACY FIX C: style-insensitive exact match (reported alongside strict EM) ----
# Sinhala answers end in a grammatical copula/particle that carries no content: the model
# writes "රෝලට් පනත යි." where the reference says "රෝලට් පනත ය." — token F1 scores that 1.00
# (the particles are stopwords) but strict EM calls it a miss. Stripping the trailing
# particle measures whether the CONTENT matched. Strict EM is still reported first and is
# what stays comparable with the Llama v6 numbers; this is an additional diagnostic, not a
# replacement.
COPULA_SUFFIXES = (
    " ලෙස ය", " යනුවෙනි", " වශයෙනි", " ලෙසිනි", " විසිනි",
    " වේ", " යි", " ය", "යනුවෙනි", "වශයෙනි", "ලෙසිනි",
)


def strip_copula(value):
    text = normalize_answer(value)
    changed = True
    while changed:
        changed = False
        for suffix in COPULA_SUFFIXES:
            if text.endswith(suffix) and len(text) > len(suffix):
                text = text[: -len(suffix)].strip(" ,.।")
                changed = True
                break
    return text


def style_insensitive_match(prediction, reference):
    return strip_copula(prediction) == strip_copula(reference)


def grounded_prefix(text, context, run=UNGROUNDED_RUN_TO_CUT):
    """Keep the prefix of `text` before the first run of `run` consecutive ungrounded words.

    Operates on whitespace-separated words, so Sinhala abbreviations that embed full stops
    (`ක්‍රි.ව.`, `සී. ඩබ්ලිව්.`) survive intact — unlike splitting on '.', which would cut
    them apart. Validated on the previous run's outputs: raises token F1 on attempted rows
    from 0.275 to 0.559.
    """
    if run <= 0:
        return clean_text(text)
    normalized_context = " ".join(lexical_tokens(context))
    words = clean_text(text).split()
    consecutive_bad = 0
    cut = len(words)
    for index, word in enumerate(words):
        tokens = lexical_tokens(word)
        ok = (not tokens) or any(token_supported(t, normalized_context) for t in tokens)
        consecutive_bad = 0 if ok else consecutive_bad + 1
        if consecutive_bad >= run:
            cut = index - run + 1
            break
    return " ".join(words[: max(cut, 1)])


def generate_answer(context, question):
    prompt = render_prompt(context, question)
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
    generate_kwargs = dict(
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,                 # deterministic either way
        repetition_penalty=1.05,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
        use_cache=True,
    )
    if NUM_BEAMS > 1:                    # ACCURACY FIX A
        generate_kwargs.update(
            num_beams=NUM_BEAMS,
            length_penalty=LENGTH_PENALTY,
            early_stopping=True,
        )
    with torch.inference_mode():
        output_ids = model.generate(**inputs, **generate_kwargs)
    generated = output_ids[0, inputs["input_ids"].shape[-1] :]
    ids = generated.tolist()
    emitted_eos = tokenizer.eos_token_id in ids          # FIX 4: is it terminating?
    text = tokenizer.decode(ids, skip_special_tokens=True).strip()
    text = text.splitlines()[0].strip(" []{}()<>\"'`") if text else ""
    return {"raw_answer": text, "emitted_eos": emitted_eos, "generated_tokens": len(ids)}


def run_qa(context, question, use_grounding=USE_GROUNDING):
    result = generate_answer(context, question)
    raw = result["raw_answer"]

    # FIX 5 + 6: clean first, THEN score the gate on the cleaned answer. The previous run
    # scored support over the whole rambling generation, so filler dragged correct answers
    # below threshold and the gate rewrote them as refusals (50.4% of answerable rows).
    cleaned = raw if is_no_answer(raw) else grounded_prefix(raw, context)
    was_trimmed = cleaned != raw

    if is_no_answer(cleaned) or not cleaned:
        final, support, gated = NO_ANSWER, 1.0, False
    else:
        support = evidence_support(cleaned, context)
        if use_grounding and support < GROUNDING_THRESHOLD:
            final, gated = NO_ANSWER, True
        else:
            final, gated = cleaned, False

    return {
        "answer": final,
        "raw_answer": raw,
        "cleaned_answer": cleaned,
        "support": support,
        "gated": gated,
        "trimmed": was_trimmed,
        "emitted_eos": result["emitted_eos"],
        "generated_tokens": result["generated_tokens"],
    }


def token_f1(prediction, reference):
    pd = re.findall(r"\d+", normalize_answer(prediction))
    rd = re.findall(r"\d+", normalize_answer(reference))
    if rd and pd != rd:
        return 0.0
    pt, rt = lexical_tokens(prediction), lexical_tokens(reference)
    if not pt and not rt:
        return 1.0
    if not pt or not rt:
        return 0.0
    overlap = sum((Counter(pt) & Counter(rt)).values())
    if not overlap:
        return 0.0
    precision, recall = overlap / len(pt), overlap / len(rt)
    return 2 * precision * recall / (precision + recall)


print("Inference helpers ready.")

In [ ]:
test_records = load_jsonl(TEST_PATH)
print(Counter(r["answerable"] for r in test_records))

row = test_records[0]
smoke = run_qa(row["context"], row["question"])
print("\n--- smoke test ---")
print("Question   :", row["question"])
print("Expected   :", canonical_answer(row))
print("Raw        :", smoke["raw_answer"])
print("Cleaned    :", smoke["cleaned_answer"])
print("Final      :", smoke["answer"])
print("Emitted EOS:", smoke["emitted_eos"], "| tokens:", smoke["generated_tokens"],
      "| support:", f"{smoke['support']:.3f}")

In [ ]:
predictions = []
transcript = []

for index, item in enumerate(test_records, 1):
    reference = canonical_answer(item)
    result = run_qa(item["context"], item["question"])
    prediction = result["answer"]

    exact = normalize_answer(prediction) == normalize_answer(reference)
    style_exact = style_insensitive_match(prediction, reference)
    f1 = token_f1(prediction, reference)

    predictions.append({
        "index": index,
        "grade": item.get("grade"),
        "chapter": item.get("chapter"),
        "answerable": item["answerable"],
        "question": item["question"],
        "reference": reference,
        "raw_prediction": result["raw_answer"],
        "cleaned_prediction": result["cleaned_answer"],
        "prediction": prediction,
        "exact_match": exact,
        "style_insensitive_match": style_exact,
        "token_f1": f1,
        "evidence_support": result["support"],
        "gated": result["gated"],
        "trimmed": result["trimmed"],
        "emitted_eos": result["emitted_eos"],
        "generated_tokens": result["generated_tokens"],
    })

    block = [
        "",
        "=" * 100,
        f"[{index}/{len(test_records)}]",
        f"Answerable: {item['answerable']}",
        f"Question : {item['question']}",
        f"Expected : {reference}",
        f"Generated: {prediction}",
        f"Exact/F1 : {exact} / {f1:.3f}",
    ]
    print("\n".join(block), flush=True)
    transcript.extend(block)

with RESULTS_JSONL.open("w", encoding="utf-8", newline="\n") as handle:
    for p in predictions:
        handle.write(json.dumps(p, ensure_ascii=False) + "\n")

print("\nFinished evaluation.")

In [ ]:
total = len(predictions)
answerable = [p for p in predictions if p["answerable"]]
unanswerable = [p for p in predictions if not p["answerable"]]

exact_correct = sum(p["exact_match"] for p in predictions)
style_correct = sum(p["style_insensitive_match"] for p in predictions)
f1_mean = sum(p["token_f1"] for p in predictions) / total
ans_exact = sum(p["exact_match"] for p in answerable)
una_exact = sum(p["exact_match"] for p in unanswerable)
false_answers = sum(1 for p in unanswerable if not is_no_answer(p["prediction"]))
over_refusal = sum(1 for p in answerable if is_no_answer(p["prediction"]))

# FIX 4: termination health — the metric that would have caught the previous failure.
eos_rate = sum(p["emitted_eos"] for p in predictions) / total
trimmed = sum(p["trimmed"] for p in predictions)
gated = sum(p["gated"] for p in predictions)
hit_cap = sum(1 for p in predictions if p["generated_tokens"] >= MAX_NEW_TOKENS)

SUMMARY = [
    "",
    "=" * 100,
    "EXTERNAL TEST RESULTS",
    "=" * 100,
    f"Model                : CPT ({CPT_ADAPTER_ID}) + QA fine-tune",
    f"Test file            : {TEST_PATH}",
    f"Decoding             : {'beam search x' + str(NUM_BEAMS) if NUM_BEAMS > 1 else 'greedy'}"
    f" | gate threshold {GROUNDING_THRESHOLD} | epochs {NUM_EPOCHS}",
    "",
    f"Exact match          : {exact_correct}/{total} ({100*exact_correct/total:.2f}%)",
    f"Style-insensitive EM : {style_correct}/{total} ({100*style_correct/total:.2f}%)",
    f"Mean token F1        : {f1_mean:.4f}",
    f"Answerable exact     : {ans_exact}/{len(answerable)} ({100*ans_exact/max(len(answerable),1):.2f}%)",
    f"Unanswerable exact   : {una_exact}/{len(unanswerable)} ({100*una_exact/max(len(unanswerable),1):.2f}%)",
    f"False-answer rate    : {false_answers}/{len(unanswerable)} ({100*false_answers/max(len(unanswerable),1):.2f}%)",
    f"Over-refusal (answ.) : {over_refusal}/{len(answerable)} ({100*over_refusal/max(len(answerable),1):.2f}%)",
    "",
    "--- termination health (the previous run's failure mode) ---",
    f"Emitted EOS          : {100*eos_rate:.1f}%  (target: ~100%; low means it still does not stop)",
    f"Hit the token cap    : {hit_cap}/{total}",
    f"Trimmed by grounding : {trimmed}/{total}  (how often the safety net had to fire)",
    f"Gated to refusal     : {gated}/{total}",
    "",
    "--- reference points on this split ---",
    "Qwen3-4B-Instruct-2507 (stock)      : EM 20.26% | F1 0.5048 | unans 77.8% | over-refusal  0.0%",
    "qwen3-4b-sinhala-qa-poc (no CPT)    : EM 20.13% | F1 0.5015 | unans 77.8% | over-refusal  0.0%",
    "qwen3-4b-sinhala-qa-cpt v1 (broken) : EM 12.42% | F1 0.2414 | unans  100% | over-refusal 50.4%",
    "qwen3-4b-sinhala-qa-cpt v2          : EM 28.76% | F1 0.6161 | unans  100% | over-refusal  6.7%",
    "SinLlama 3B (Llama route, for scale): EM 32.03% | F1 0.6504",
    "=" * 100,
]
for line in SUMMARY:
    print(line)
transcript.extend(SUMMARY)

with RESULTS_TXT.open("w", encoding="utf-8", newline="\n") as handle:
    handle.write("\n".join(transcript) + "\n")
print(f"\nSaved: {RESULTS_JSONL}\n       {RESULTS_TXT}")

In [ ]:
# ---- Diagnostics: did the fixes work? ----
print("If EOS rate is high and 'trimmed' is low, termination was learned and the safety")
print("net is idle — the intended outcome. If EOS rate is low but scores are good, the")
print("safety net is carrying the result and the model still has not learned to stop.\n")

not_terminated = [p for p in predictions if not p["emitted_eos"]]
print(f"Generations that never emitted EOS: {len(not_terminated)}")
for p in not_terminated[:5]:
    print("-" * 95)
    print("Q   :", p["question"][:90])
    print("Raw :", p["raw_prediction"][:160])
    print("Cut :", p["cleaned_prediction"][:160])

low_f1 = [p for p in predictions if p["answerable"] and p["token_f1"] < 0.5]
print(f"\nAnswerable rows below 0.5 F1: {len(low_f1)}")
for p in low_f1[:5]:
    print("-" * 95)
    print("Q   :", p["question"][:90])
    print("Ref :", p["reference"][:90])
    print("Pred:", p["prediction"][:120], f"| support {p['evidence_support']:.3f}")

In [ ]:
# ---- Push to Hugging Face ----
from huggingface_hub import HfApi

api = HfApi(token=os.environ["HF_TOKEN"])
print(f"Pushing QA adapter to {ADAPTER_REPO_ID}...")
api.create_repo(ADAPTER_REPO_ID, private=HF_REPO_PRIVATE, exist_ok=True)
api.upload_folder(
    repo_id=ADAPTER_REPO_ID,
    folder_path=str(QA_ADAPTER_DIR),
    commit_message="Qwen3-4B Sinhala QA LoRA on the CPT checkpoint (termination-corrected)",
)
print(f"  -> https://huggingface.co/{ADAPTER_REPO_ID}")

print(f"Pushing merged model to {MERGED_REPO_ID}...")
model.push_to_hub(
    MERGED_REPO_ID,
    token=os.environ["HF_TOKEN"],
    safe_serialization=True,
    max_shard_size="5GB",
    private=HF_REPO_PRIVATE,
    commit_message="Qwen3-4B Sinhala CPT + QA fine-tune (termination-corrected)",
)
tokenizer.push_to_hub(
    MERGED_REPO_ID,
    token=os.environ["HF_TOKEN"],
    private=HF_REPO_PRIVATE,
    commit_message="Extended Sinhala tokenizer",
)
print(f"  -> https://huggingface.co/{MERGED_REPO_ID}")

## Reading the result

The headline number to check first is **Emitted EOS**, not exact match. It separates the
two possible outcomes:

- **EOS ~100%, few trims** — termination was actually learned. The scores are the model's
  own, and the grounded-prefix net is idle. This is the outcome to report.
- **EOS low, scores still decent** — the safety net is doing the work and the model has
  not learned to stop. The result is real but should be described as *inference-time
  mitigation*, not a fixed model. In that case the honest next step is more epochs, or
  accepting that ~180M tokens of raw-text CPT needs more than 1,375 QA examples to undo.

Set expectations accordingly: the CPT run behind `isji/qwen3-4b-cpt` stopped at 2,740 of
3,744 planned steps (73%) with its cosine schedule incomplete, so its 25,187 new
embedding rows are under-trained. **Parity with stock Qwen (F1 ≈ 0.50) is a success
here** — it would mean the CPT arm works and the tokenizer efficiency gain (1.48 vs 9.19
tokens/word) comes for free at equal accuracy. Beating stock outright would need the CPT
re-run to completion.